In [5]:
import pyautogui

In [55]:
pyautogui.mouseInfo()

In [23]:
game_region = (42, 277, 802, 484)

In [64]:
screenshot = pyautogui.screenshot(region=game_region)

In [65]:
screenshot.save("./apple.jpg")

In [5]:
import base64

In [7]:
with open(r"C:\Users\playdata2\apple.jpg", 'rb') as image_file:
    tmp_base = base64.b64encode(image_file.read()).decode('utf-8')

In [8]:
from openai import OpenAI
client = OpenAI()

In [10]:
response = client.responses.create(
    model="gpt-4o-mini-2024-07-18",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "첨부된 이미지가 어떤 이미지인가?"},
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{tmp_base}" },
        ],
    }],
)


In [11]:
response.output_text

'제공하신 이미지는 여러 개의 빨간 사과 안에 숫자가 적힌 배열입니다. 이는 종종 관찰력이나 집중력 테스트용 퍼즐 또는 게임에서 사용되는 형태로, 특정 숫자를 찾거나 패턴을 인식하는 용도로 활용될 수 있습니다.'

In [12]:
prompt = """
첨부된 이미지는 게임의 화면이다. 각각의 숫자를 배열로 보고,
이 배열에서 가로 및 세로로 인접한 원소들ㄹ로 이루어진 직사각형 부분 배열 중
내부 원소들의 합이 정확히 10이 되는 모든 경우를 찾아라.
"""
response = client.responses.create(
    model="gpt-4o-mini-2024-07-18",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": prompt},
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{tmp_base}" },
        ],
    }],
)




print(response.output_text)


주어진 배열에서 내부 원소들의 합이 정확히 10이 되는 모든 직사각형 부분 배열을 찾아보겠습니다. 

1. **배열을 분석**: 배열의 각 원소를 가로 및 세로로 탐색하며 부분 배열의 합을 계산해야 합니다.

2. **부분 배열 생성**: 각 시작점에서 가능한 모든 끝점까지의 직사각형을 만들어 원소의 합을 구합니다.

3. **합 계산**: 부분 배열의 원소들을 모두 더하여 10이 되는 경우를 기록합니다.

아래는 사이클과 내적합을 통해 구현하는 일반적인 방법입니다:

```python
def find_rectangles_with_sum_10(grid):
    rows = len(grid)
    cols = len(grid[0])
    rectangles = []

    for r1 in range(rows):
        for r2 in range(r1, rows):
            for c1 in range(cols):
                for c2 in range(c1, cols):
                    total = 0
                    for r in range(r1, r2 + 1):
                        for c in range(c1, c2 + 1):
                            total += grid[r][c]
                    if total == 10:
                        rectangles.append((r1, c1, r2, c2))
    
    return rectangles

# 예제 배열
grid = [
    [9, 7, 9, 8, 9, 5, 3, 5, 8, 2, 3, 4, 7, 6],
    [5, 1, 1, 5, 4, 6, 4, 2, 1, 6, 5, 2, 9, 1],
    [3, 5, 7, 8, 6, 2, 9, 9, 1, 4, 5, 8, 5, 5],
    [4, 7, 5, 2, 

In [21]:
from pydantic import BaseModel, Field
from typing import List
class GameBoard(BaseModel):
    board: List[List[int]] = Field(
        description="사과 게임의 숫자 격자판, 각 행의 숫자들을 담은 2차원 정수 배열이다."
    )


model_name = "gpt-4o-mini-2024-07-18"
prompt = """
첨부된 이미지는 숫자가 적힌 사과드이 격자 형태로 배치된 게임 화면이다.
화면에서 숫자들만 정확히 읽어내어 2차원 배열 형태로 반환하라.
"""
response = client.beta.chat.completions.parse(
    model = model_name,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url":  
                 {"url" : f"data:image/jpeg;base64,{tmp_base}"}
            },
        ],
    }],
    response_format=GameBoard,
    temperature=0.0
)

board = response.choices[0].message.parsed
board


GameBoard(board=[[9, 7, 9, 8, 9, 5, 3, 8, 5, 8, 2, 3, 7, 6], [5, 1, 1, 5, 4, 6, 4, 9, 1, 2, 1, 6, 5, 2], [3, 5, 7, 8, 6, 2, 9, 9, 1, 4, 1, 5, 8, 5], [4, 7, 3, 5, 2, 9, 9, 1, 4, 5, 2, 6, 3, 5], [5, 1, 2, 9, 3, 2, 7, 3, 6, 1, 3, 5, 5, 5], [7, 7, 1, 4, 1, 2, 6, 4, 6, 7, 5, 6, 4, 3], [6, 6, 4, 3, 5, 5, 9, 1, 5, 1, 5, 9, 1, 1], [4, 6, 1, 4, 6, 3, 8, 1, 1, 3, 7, 9, 1, 1], [7, 3, 1, 2, 1, 5, 9, 5, 7, 4, 8, 1, 6, 3], [7, 1, 9, 3, 9, 1, 6, 8, 7, 5, 3, 5, 5, 3]])

In [24]:
class MoveSolution(BaseModel):
    moves : List[List[List[int]]] = Field(
        description="합이 10이 되는 직사각형 영역의 시작 좌표 [r1, c1]와 끝 좌표 [r2, c2]를 묶은 배열들의 목록이다. 예시: [[[0,0], [0,3]], [[1,2], [2,2]]]"
    )




prompt = f"""
다음은 숫자로 이루어진 2차원 배열이다. \n\n{board}\n\n
다음 정보는 화면의 시작 x 좌표값,y 좌표값, x 좌표값에서 상대적인 가로길이, y 좌표값에서 상대적인 세로 길이 정보이다. \n\n{game_region}\n\n
이 배열값과 화면 좌표값을 이용해서 가로 및 세로로 인접한 원소들로 이루어진 직사각형 부분 배열 중 내부 원소들의 합이 정확히 10이되는 모든 경우를 찾아라.  결과는 제공된 스키마에 맞추어 반환할 것
반듯이 화면 좌표값으로만 출력할 것
"""
resp = client.beta.chat.completions.parse(
    model = model_name ,
    messages = [
        {'role' : 'user', 'content' : prompt}
    ],
    response_format=MoveSolution,
    temperature=0.0
)
